# M3-CLV 구성비 적응형 가치그래프 — Dunnhumby seed 42 validation

M1과 전체 CLV 구성비 적응형 M3만 비교합니다. 학습구간의 $L_u=\log(1+N_u)+\log(1+V_u)$와 각 고객의 로그 N/V 구성비로 엣지 전파가중치를 만듭니다. **먼저 별도 N 세그먼트 진단 노트북에서 `proceed_to_compositional_m3=True`를 확인한 경우에만 실행하세요.** test와 holdout은 만들거나 평가하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = '4322476cb2d51ace30266b21dee1ddb62f030a84'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('검토 코드 고정 완료:', actual_sha)

In [ ]:
import json, torch
from lightgcn_clv_m3_composition import (
    configure_m3_clv_composition_dunnhumby_run,
    preflight_summary,
    run_experiment,
)

cfg = configure_m3_clv_composition_dunnhumby_run()
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_experiment(cfg)

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'revenue@10', 'revenue@20', 'revenue@50', 'arp@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'value_alignment',
]
available = [column for column in columns if column in result_df.columns]
validation = result_df[result_df['split'].eq('val')][available]
display(validation.sort_values('model_id'))
print('screening 판정:')
print(json.dumps(result_df.attrs['screening_decision'], ensure_ascii=False, indent=2))
print('결과 폴더:', result_df.attrs['out_dir'])